# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform

REPO_DIR = "K4-Track4-Day2-NgoTheViet-02594"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TheViet298/K4-Track4-Day2-NgoTheViet-02594.git

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/submissions/02594_NgoTheViet/code")

import numpy as np, pandas as pd, torch, timm

print('Python :', platform.python_version())
print('PyTorch:', torch.__version__)
print('timm   :', timm.__version__)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO GPU')


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
# Tự động tìm đường dẫn IMAGES_DIR và LABELS_DIR (dù lồng bao nhiêu cấp trong /kaggle/input)
import os, glob

# 1. Tìm thư mục nhãn (nơi chứa train_subset0.csv)
csv_matches = glob.glob('/kaggle/input/**/train_subset0.csv', recursive=True)
if csv_matches:
    LABELS_DIR = os.path.dirname(csv_matches[0])
elif os.path.exists('data/labels'):
    LABELS_DIR = 'data/labels'
else:
    LABELS_DIR = 'data/labels'

# 2. Tìm thư mục ảnh (nơi chứa các file .jpg)
img_dir_matches = glob.glob('/kaggle/input/**/images', recursive=True)
if img_dir_matches and os.path.isdir(img_dir_matches[0]):
    IMAGES_DIR = img_dir_matches[0]
else:
    jpg_matches = glob.glob('/kaggle/input/**/*.jpg', recursive=True)
    if jpg_matches:
        IMAGES_DIR = os.path.dirname(jpg_matches[0])
    elif os.path.exists('data/images'):
        IMAGES_DIR = 'data/images'
    elif os.path.exists('images'):
        IMAGES_DIR = 'images'
    else:
        IMAGES_DIR = 'data/images'

print(f"IMAGES_DIR: {IMAGES_DIR}")
print(f"LABELS_DIR: {LABELS_DIR}")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# 1. Đọc split bằng dataset.load_split, chạy dataset.check_split(...)
import os
import dataset as ds
import matplotlib.pyplot as plt

train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
info = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)

print("=== SỐ LIỆU TẬP DỮ LIỆU FOLD 0 ===")
print(f"Train: {info['n']['train']} ảnh ({info['n']['percentages']['train']:.2f}%)")
print(f"Val:   {info['n']['val']} ảnh ({info['n']['percentages']['val']:.2f}%)")
print(f"Test:  {info['n']['test']} ảnh ({info['n']['percentages']['test']:.2f}%)")
print(f"Tổng:  {info['n']['total']} ảnh | Giao rỗng: OK | Hợp 17509: OK")

# 2. Biểu đồ phân bố lớp (map từ Label int sang tên loài CLASS_NAMES)
species_counts = train_df['Label'].map(lambda x: ds.CLASS_NAMES[x]).value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
species_counts.plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.set_title("Phân bố số lượng ảnh theo loài (Tập Train Fold 0)")
ax.set_ylabel("Số lượng ảnh")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 3. Trực quan hóa ảnh mẫu
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, name in enumerate(ds.CLASS_NAMES):
    row, col = i // 3, i % 3
    sample_rows = train_df[train_df['Label'] == i]
    if len(sample_rows) > 0:
        sample_fn = sample_rows['Filename'].iloc[0]
        img = plt.imread(os.path.join(IMAGES_DIR, sample_fn))
        axes[row, col].imshow(img)
    axes[row, col].set_title(f"{i}: {name}", fontsize=10)
    axes[row, col].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Kiểm tra pipeline (slide tr. 59, GUIDE.md mục 1.3)
import os
import numpy as np
import torch
import torch.nn.functional as F
import model as model_module
import train as train_module
import dataset as ds
import matplotlib.pyplot as plt

# 1. Cố định seed
train_module.set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Loss ban đầu xấp xỉ -ln(1/9) ≈ 2.197
m = model_module.build_model("resnet50", pretrained=True, num_classes=9).to(device)
m.eval()
dummy_x = torch.randn(8, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (8,), device=device)
with torch.no_grad():
    init_loss = F.cross_entropy(m(dummy_x), dummy_y).item()
expected_loss = np.log(9)
print(f"1) Loss ban đầu của ResNet-50: {init_loss:.4f} (Kỳ vọng ≈ -ln(1/9) = {expected_loss:.4f})")
assert abs(init_loss - expected_loss) < 0.6, f"Loss ban đầu lệch quá xa: {init_loss}"

# 3. Overfit một batch nhỏ tới loss gần 0
print("2) Đang thử overfit một batch nhỏ (4 mẫu)...")
m.train()
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
x_small = torch.randn(4, 3, 224, 224, device=device)
y_small = torch.tensor([0, 1, 2, 8], device=device)
for step in range(60):
    opt.zero_grad()
    loss = F.cross_entropy(m(x_small), y_small)
    loss.backward()
    opt.step()
final_overfit_loss = loss.item()
print(f"   Loss sau 60 bước overfit: {final_overfit_loss:.6f} -> PASS (loss < 0.05)")
assert final_overfit_loss < 0.05, f"Không overfit được batch nhỏ! Loss: {final_overfit_loss}"

# 4. Trực quan ảnh sau transform (đã giải chuẩn hóa)
tf = ds.build_transforms(train=True, img_size=224, aug="basic")
dataset_sample = ds.DeepWeedsDataset(train_df.head(4), IMAGES_DIR, transform=tf)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
mean = torch.tensor(ds.IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(ds.IMAGENET_STD).view(3, 1, 1)
for idx in range(4):
    img_t, lbl, fn = dataset_sample[idx]
    unnorm = (img_t * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()
    axes[idx].imshow(unnorm)
    axes[idx].set_title(f"{ds.CLASS_NAMES[lbl]}", fontsize=10)
    axes[idx].axis("off")
plt.tight_layout()
plt.show()
print("=== PIPELINE SANITY CHECKS HOÀN TẤT VÀ ĐẠT 100% ===")


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# BƯỚC 1: SO SÁNH 5 BACKBONE VỚI CÔNG THỨC NỀN T00 (12 epochs, batch 64, seed 0)
import train as train_module
import pandas as pd

backbones_to_run = [
    ("B01", "resnet50", "ResNet default baseline"),
    ("B02", "resnext50_32x4d", "ResNeXt"),
    ("B03", "deit_small_patch16_224", "DeiT-S Transformer"),
    ("B04", "efficientnet_b0", "EfficientNet-B0 (mạng nhẹ)"),
    ("B05", "mobilenetv3_large_100", "MobileNetV3-L (mạng nhẹ)")
]

b_results = []
for exp_id, bb_name, desc in backbones_to_run:
    print(f"\n{'='*50}")
    print(f"BẮT ĐẦU CHẠY THÍ NGHIỆM {exp_id}: {bb_name} ({desc})")
    print(f"{'='*50}")
    cfg = train_module.Config(
        exp_id=exp_id,
        backbone=bb_name,
        init="finetune",
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        seed=0
    )
    res = train_module.run(cfg)
    res["description"] = desc
    b_results.append(res)

df_backbones = pd.DataFrame(b_results)
df_backbones.to_csv("backbones_summary.csv", index=False)
display(df_backbones[["exp_id", "backbone", "n_params_M", "gmacs", "best_val_macro_f1", "avg_epoch_time_sec"]])


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# BƯỚC 2: KHẢO SÁT CÔNG THỨC HUẤN LUYỆN (ABLATION TRÊN >= 3 TRỤC)
import train as train_module
import pandas as pd

# Chọn backbone tốt nhất (ResNet-50) để khảo sát các trục A, B, C, D
recipes = [
    ("T01", {"init": "scratch"}, "Trục A: Khởi tạo từ đầu (scratch)"),
    ("T02", {"init": "frozen"}, "Trục A: Đóng băng backbone, chỉ train head"),
    ("T03", {"aug": "color"}, "Trục B: Augmentation đổi màu (ColorJitter)"),
    ("T04", {"aug": "randaug"}, "Trục B: RandAugment"),
    ("T05", {"mix": "cutmix", "mix_alpha": 1.0}, "Trục B: CutMix"),
    ("T06", {"loss": "ls", "label_smoothing": 0.1}, "Trục C: Label Smoothing (eps=0.1)"),
    ("T07", {"loss": "focal", "focal_gamma": 2.0}, "Trục C: Focal Loss (gamma=2.0)"),
    ("T08", {"loss": "ce_weighted", "class_weight_beta": 0.0}, "Trục C: CE có trọng số nghịch đảo tần suất lớp"),
    ("T09", {"sampler": "balanced"}, "Trục D: WeightedRandomSampler cân bằng lớp")
]

t_results = []
for exp_id, params, desc in recipes:
    print(f"\n{'='*50}")
    print(f"BẮT ĐẦU CHẠY {exp_id}: {desc}")
    print(f"{'='*50}")
    cfg = train_module.Config(
        exp_id=exp_id,
        backbone="resnet50",
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        seed=0,
        **params
    )
    res = train_module.run(cfg)
    res["description"] = desc
    t_results.append(res)

df_training = pd.DataFrame(t_results)
df_training.to_csv("training_summary.csv", index=False)
display(df_training[["exp_id", "description", "best_val_macro_f1"]])


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# BƯỚC 3: CÁC PHƯƠNG PHÁP SUY LUẬN VÀ ĐO ĐỘ TRỄ CHUẨN XÁC
import os, torch, numpy as np, pandas as pd
import inference as inf_module
import benchmark as bench_module
import dataset as ds_module
import model as model_module
import eval as eval_module

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Nạp model tốt nhất (ví dụ B01_resnet50)
best_model = model_module.build_model("resnet50", pretrained=False, num_classes=9).to(device)
ckpt = torch.load("runs/B01/seed0/best_checkpoint.pt", map_location=device)
best_model.load_state_dict(ckpt["model_state_dict"])
best_model.eval()

# Loader val
val_tf = ds_module.build_transforms(train=False, img_size=224)
val_loader = ds_module.make_loader(val_df, IMAGES_DIR, transform=val_tf, batch_size=64, train=False)

# I00: 1-view mốc (baseline)
fns, y_true, logits_i00 = inf_module.predict_logits(best_model, val_loader, device, view=None)
probs_i00 = inf_module.aggregate_views([logits_i00], space="prob")
eval_i00 = eval_module.compute_metrics(y_true, np.argmax(probs_i00, -1), probs_i00)

# I01: TTA 2-view (Gốc + Lật ngang)
_, _, logits_flip = inf_module.predict_logits(best_model, val_loader, device, view=inf_module.view_hflip)
probs_tta = inf_module.aggregate_views([logits_i00, logits_flip], space="prob")
eval_tta = eval_module.compute_metrics(y_true, np.argmax(probs_tta, -1), probs_tta)

# I02: Temperature Scaling (khớp T trên VAL)
best_T = inf_module.fit_temperature(logits_i00, y_true)
probs_temp = inf_module.apply_temperature(logits_i00, best_T)
eval_temp = eval_module.compute_metrics(y_true, np.argmax(probs_temp, -1), probs_temp)

# I03: Model Fusion (Gộp BatchNorm vào Conv2d)
fused_model = inf_module.fuse_conv_bn(best_model)

# Đo độ trễ chính xác (Warmup >= 15, đo 60 lần, p50/p95/p99)
lat_fp32 = bench_module.latency_report(best_model, batch_size=1, img_size=224, dtype="fp32", device="cuda")
lat_amp = bench_module.latency_report(best_model, batch_size=1, img_size=224, dtype="amp", device="cuda")
lat_fused = bench_module.latency_report(fused_model, batch_size=1, img_size=224, dtype="fp32", device="cuda")

print(f"I00 (1-view)       : Macro-F1 = {eval_i00['macro_f1']:.4f} | ECE = {eval_i00['ece']:.4f}")
print(f"I01 (TTA 2-view)   : Macro-F1 = {eval_tta['macro_f1']:.4f} | ECE = {eval_tta['ece']:.4f}")
print(f"I02 (Temp T={best_T:.3f}): Macro-F1 = {eval_temp['ece']:.4f} | ECE = {eval_temp['ece']:.4f}")
print(f"Độ trễ FP32 batch 1: p50={lat_fp32['p50_ms']}ms, p95={lat_fp32['p95_ms']}ms")
print(f"Độ trễ Gộp BN     : p50={lat_fused['p50_ms']}ms, p95={lat_fused['p95_ms']}ms")


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# BƯỚC 4: CHUNG KẾT (>= 3 SEED TRÊN TEST CHO CẢ CẤU HÌNH CUỐI VÀ MỐC T00)
import train as train_module

# Chạy cấu hình chung kết (F01) và mốc (T00) trên 3 seed: 0, 1, 2
for seed in (0, 1, 2):
    print(f"\n{'='*50}")
    print(f"CHUNG KẾT F01 (Seed {seed}) - ĐÁNH GIÁ TRÊN TEST")
    print(f"{'='*50}")
    train_module.run(train_module.Config(
        exp_id="F01",
        backbone="resnet50",
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        seed=seed,
        save_test_predictions=True
    ))

    print(f"\n{'='*50}")
    print(f"MỐC SO SÁNH T00 (Seed {seed}) - ĐÁNH GIÁ TRÊN TEST")
    print(f"{'='*50}")
    train_module.run(train_module.Config(
        exp_id="T00",
        backbone="resnet50",
        epochs=12,
        batch_size=64,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        seed=seed,
        save_test_predictions=True
    ))
print("=== HOÀN TẤT CÁC LẦN CHẠY CHUNG KẾT VÀ MỐC ===")


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# BƯỚC 5: XUẤT TOÀN BỘ KẾT QUẢ RA results.xlsx
import os, glob, json, pandas as pd

xlsx_path = "results.xlsx"
writer = pd.ExcelWriter(xlsx_path, engine="openpyxl")

# 1. Sheet Backbones
if os.path.exists("backbones_summary.csv"):
    df_b = pd.read_csv("backbones_summary.csv")
    df_b.to_excel(writer, sheet_name="Backbones", index=False)

# 2. Sheet Training
if os.path.exists("training_summary.csv"):
    df_t = pd.read_csv("training_summary.csv")
    df_t.to_excel(writer, sheet_name="Training", index=False)

# 3. Sheet Summary
summary_rows = []
for s_file in glob.glob("runs/**/summary.json", recursive=True):
    with open(s_file) as f:
        summary_rows.append(json.load(f))
if summary_rows:
    df_sum = pd.DataFrame(summary_rows)
    df_sum.to_excel(writer, sheet_name="Summary", index=False)

writer.close()
print(f"✅ Đã tạo file thành công: {xlsx_path}")
